In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Tier-2 leaf libs pinned exactly. Tier-1 (torch/CUDA/python) inherited from the
# pinned Kaggle image — do NOT pip-install torch here or you'll break the GPU.
#%pip install -q "sae-lens==6.44.2" "transformer-lens==2.18.0" "sae-dashboard==0.7.2" "numpy==1.26.4"

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/tmp/hf"          # ephemeral scratch, not /content
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
import torch
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
# Fail fast if the leaf install disturbed tier-1:
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", device)
assert device == "cuda", "GPU not visible — check Accelerator setting / install didn't swap torch"


In [ ]:
from sae_lens import SAE, HookedSAETransformer

# This avoids staging model in CPU RAM during load
model = HookedSAETransformer.from_pretrained(
    "google/gemma-2-2b",
    dtype=torch.float32,
    device=device,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    move_to_device=True,
)

model.eval()
print(f"n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}")

In [ ]:
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id="layer_20/width_16k/canonical",
    device=device
)
sae = sae.to(torch.bfloat16)

allocated = torch.cuda.memory_allocated() / 1e9
print(f"GPU allocated: {allocated:.1f}GB / 15GB")
print(f"SAE d_sae: {sae.cfg.d_sae}, d_in: {sae.cfg.d_in}")

In [ ]:
#!pip freeze > /kaggle/working/requirements-lock.txt

The "sae" object is an instance of the SAE (Sparse Autoencoder class). There are many different SAE architectures which may have different weights or activation functions. In order to simplify working with SAEs, SAE Lens handles most of this complexity for you.

Let's look at the SAE config and understand each of the parameters:

1. `architecture`: Specifies the type of SAE architecture being used, in this case, the standard architecture (encoder and decoder with hidden activations, as opposed to a gated SAE).
2. `d_in`: Defines the input dimension of the SAE, which is [768] in this configuration.
3. `d_sae`: Sets the dimension of the SAE's hidden layer, which is [24576] here. This represents the number of possible feature activations.
4. `activation_fn_str`: Specifies the activation function used in the SAE, which is ReLU in this case. TopK is another option that we will not cover here.
5. `apply_b_dec_to_input`: Determines whether to apply the decoder bias to the input, set to True here.
6. `finetuning_scaling_factor`: Indicates whether to use a scaling factor to weight initialization and the forward pass. This is not usually used and was introduced to support a [solution for shrinkage](https://www.lesswrong.com/posts/3JuSjTZyMzaSeTxKk/addressing-feature-suppression-in-saes).
7. `context_size`: Defines the size of the context window, which is [128] tokens in this case. In turns out SAEs trained on small activations from small prompts [often don't perform well on longer prompts](https://www.lesswrong.com/posts/baJyjpktzmcmRfosq/stitching-saes-of-different-sizes).
8. `model_name`: Specifies the name of the model being used, which is 'gemma-2-2b' here. [This is a valid model name in TransformerLens](https://transformerlensorg.github.io/TransformerLens/generated/model_properties_table.html).
9. `hook_name`: Indicates the specific hook in the model where the SAE is applied.
10. `hook_head_index`: Defines which attention head to hook into; not relevant here since we are looking at a residual stream SAE.
11. `prepend_bos`: Determines whether to prepend the beginning-of-sequence token, set to True.
12. `dataset_path`: Specifies the path to the dataset used for training or evaluation. (Can be local or a huggingface dataset.)
13. `dataset_trust_remote_code`: Indicates whether to trust remote code (from HuggingFace) when loading the dataset, set to True.
14. `normalize_activations`: Specifies how to normalize activations, set to 'none' in this config.
15. `dtype`: Defines the data type for tensor operations, set to 32-bit floating point.
16. `device`: Specifies the computational device to use.
17. `sae_lens_training_version`: Indicates the version of SAE Lens used for training, set to None here.
18. `activation_fn_kwargs`: Allows for additional keyword arguments for the activation function. This would be used if e.g. the `activation_fn_str` was set to `topk`, so that `k` could be specified.


In [ ]:
print(sae.cfg.__dict__)

# check

In [ ]:
def get_feature_acts(prompt: str, layer: int = 20):
      tokens = model.to_tokens(prompt)
      _, cache = model.run_with_cache(
          tokens,
          names_filter=f"blocks.{layer}.hook_resid_post",
          return_type=None,
      )
      resid = cache[f"blocks.{layer}.hook_resid_post"].squeeze(
  0).to(device).float()
      feature_acts = sae.encode(resid)          # fp32 in → fp32 out
      print("max act:", feature_acts.max().item(), "| nonzero:",  (feature_acts > 0).sum().item())
      return feature_acts, model.to_str_tokens(prompt)

acts, str_tokens = get_feature_acts("The quick brown fox jumps over the lazy dog.")
l0_per_token = (acts > 0).sum(dim=1)                 # [n_tokens] count
print("tokens:   ", str_tokens)
print("L0/token: ", l0_per_token.tolist())
print("last-token L0:", l0_per_token[-1].item(), "| last-token max:", acts[-1].max().item())

In [ ]:
# load prompts
import sys, os, glob
hit = glob.glob("/kaggle/input/**/joy.py", recursive=True)
assert hit, "joy.py not found under /kaggle/input — attach the prompts dataset (Add Input)"
d = os.path.dirname(hit[0])
sys.path.insert(0, d)
from joy import JOY_PROMPTS
from neutral import NEUTRAL_PROMPTS
print("loaded from", d)
print("joy", len(JOY_PROMPTS), "neutral",
len(NEUTRAL_PROMPTS))

In [ ]:
# concept means
def concept_mean(prompts, layer=20):
      vecs = []
      for p in prompts:
          acts, _ = get_feature_acts(p, layer)
          vecs.append(acts[-1])
      return torch.stack(vecs).mean(0)

joy_mean = concept_mean(JOY_PROMPTS)
neutral_mean = concept_mean(NEUTRAL_PROMPTS)
print("joy L0", int((joy_mean > 0).sum()))
print("neutral L0", int((neutral_mean > 0).sum()))

In [ ]:
import pandas as pd

NP_URL =  "https://neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/"

def diff_table(concept_vec, baseline_vec, concept, baseline, k=20):
      diff = concept_vec - baseline_vec
      top = torch.topk(diff, k)
      rows = []
      for fid in top.indices.tolist():
          rows.append({
              "feature_id": fid,
              "diff_score": diff[fid].item(),
              concept + "_mean": concept_vec[fid].item(),
              baseline + "_mean": baseline_vec[fid].item(),
              "neuronpedia": NP_URL + str(fid),
          })
      return pd.DataFrame(rows)

diff_joy_vs_neutral = diff_table(joy_mean, neutral_mean,  "joy", "neutral")
diff_joy_vs_neutral

In [ ]:
from IPython.display import HTML

def link(fid):
      url = NP_URL + str(fid)
      return '<a href="' + url + '" target="_blank">' +   str(fid) + '</a>'

def show_links(df):
      d = df.copy()
      d["neuronpedia"] = d["feature_id"].apply(link)
      return HTML(d.to_html(escape=False))

show_links(diff_joy_vs_neutral)

In [ ]:
  import numpy as np

  fid = int(diff_joy_vs_neutral.iloc[0]["feature_id"])

  import numpy as np

  fid = int(diff_joy_vs_neutral.iloc[0]["feature_id"])

  joy_hits = []
  for p in JOY_PROMPTS:
      acts, _ = get_feature_acts(p)
      joy_hits.append(acts[-1, fid].item())

In [ ]:
# check #1 feature prompt-by prompt:
import numpy as np
fid = int(diff_joy_vs_neutral.iloc[0]["feature_id"])
joy_hits =  [get_feature_acts(p)[0][-1,  fid].item() for p in JOY_PROMPTS]
neu_hits =  [get_feature_acts(p)[0][-1,  fid].item() for p in  NEUTRAL_PROMPTS]
print(f"feature {fid}")
print(f"  joy:  mean={np.mean(joy_hits):6.2f} fires {sum(h>0 for h in  joy_hits)}/{len(joy_hits)}")
print(f"  neutral:  mean={np.mean(neu_hits):6.2f}  fires {sum(h>0 for h in   neu_hits)}/{len(neu_hits)}")
print(f"  https://neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/{fid}")